In [ ]:
install java

sudo apt-get update
sudo apt-get install openjdk-17-jre-headless

install pyspark
uv run pyspark

# Week X - Batch Processing with Apache Spark

StableFlow issuer-event data extracted from Postgres into chunked Parquet files.

**Dataset:** `data/raw_issuer_events/`

| Property | Value |
|----------|-------|
| Files | 32 Parquet files (raw_issuer_events_00000.parquet ... 00031.parquet) |
| Total rows | 991,540 |
| Total size | ~79 MB |
| Date range | 2026-05-01 to 2026-11-06 |
| Tokens | USDC, USDT |
| Event types | MINT, BURN, BLACKLIST, UNBLACKLIST, DESTROY_BLACKLIST |

## Schema (flat, one table - no normal forms)

| Column | Type | Notes |
|--------|------|-------|
| `block_number` | long | Ethereum block number |
| `block_timestamp` | timestamp (UTC, microseconds) | Block time |
| `event_date` | date | Derived from `block_timestamp` |
| `transaction_hash` | string | Lowercase, 0x prefix |
| `log_index` | int | Log index within the transaction |
| `contract_address` | string | Lowercase, 0x prefix |
| `token` | string | USDC / USDT |
| `event_type` | string | MINT, BURN, BLACKLIST, UNBLACKLIST, DESTROY_BLACKLIST |
| `topic1_address` | string | Decoded from padded topic, lowercase 0x |
| `topic2_address` | string | Decoded from padded topic, lowercase 0x |
| `amount_raw` | string | uint256 from hex data, stored as decimal string |
| `blacklisted_address` | string | Address blacklisted/unblacklisted |
| `destroyed_amount_raw` | long | Raw destroyed amount (DESTROY_BLACKLIST only) |
| `destroyed_amount_usdt` | decimal(38,6) | USD value destroyed |
| `ingested_at` | timestamp (UTC) | ETL load timestamp |

Natural key: (transaction_hash, log_index).

## Spark Cluster, Master, and local[*]

### What a Spark cluster is

A Spark application runs on a **cluster** made of two kinds of processes:

- **Driver** - the brain. It holds your Python code, builds the execution plan,
  schedules tasks, and collects results. It runs your main() or notebook cells.
- **Executors** - the muscle. They do the actual work: reading files,
  filtering rows, aggregating, writing output. Each executor has its own JVM and
  memory, and runs one or more tasks in parallel.

A cluster manager (Standalone, YARN, Kubernetes, etc.) decides which executor
gets which task and what resources they use.

### What .master(...) means

.master() tells Spark where to find the cluster manager - i.e., how to
start the driver and talk to executors. Common values:

| Master value | Meaning |
|--------------|---------|
| local | Run everything in one thread on your laptop (no parallelism). |
| local[*] | Run on your laptop using all available CPU cores as separate threads. |
| local[4] | Use exactly 4 threads. |
| spark://host:7077 | Connect to a real Spark Standalone cluster (driver + remote executors). |
| yarn / k8s | Connect to a Hadoop YARN or Kubernetes cluster. |

For this lesson we use local[*].

- It runs entirely on your machine - no separate cluster to install.
- * means use every CPU core your computer has. If you have 8 cores, Spark
  starts 8 executor threads, so it can read and process multiple Parquet files at
  once.
- You can still see task parallelism in the Spark UI: each of the 32 Parquet
  files becomes a task, and your cores run several of them simultaneously.

This is the same pattern you saw in the ZoomCamp reference notebooks
(01_spark_dataFrames.ipynb ... 06_local_cluster.ipynb), where 06 connects
to spark://DESKTOP-LQFDL79.localdomain:7077 - a real cluster instead of
local[*]. The rest of the code is identical; only the master changes.


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("stablecoin-issuer-lesson")
    .getOrCreate()
)

print("Spark UI available at http://localhost:4040")
print("Spark version:", spark.version)


Spark UI available at http://localhost:4040
Spark version: 4.3.0


## Step 1 - Read & Inspect

Load the chunked Parquet directory into a single DataFrame, inspect the schema, count rows, and show a sample.

**Spark UI checkpoint:** Open http://localhost:4040 -> Stages. You should see ~32 tasks (one per Parquet file) in the read stage - this is task parallelism in action.

In [8]:
path = "data/01_raw_data"

df = (
    spark.read
    .option("recursiveFileLookup", "true")
    .parquet(path)
)

print("=== Schema ===")
df.printSchema()

print("\nTotal rows:", f"{df.count():,}")

print("\n=== Sample ===")
df.show(5, truncate=False)


=== Schema ===
root
 |-- block_number: long (nullable = true)
 |-- block_timestamp: timestamp (nullable = true)
 |-- event_date: date (nullable = true)
 |-- transaction_hash: string (nullable = true)
 |-- log_index: integer (nullable = true)
 |-- contract_address: string (nullable = true)
 |-- token: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- topic1_address: string (nullable = true)
 |-- topic2_address: string (nullable = true)
 |-- amount_raw: string (nullable = true)
 |-- blacklisted_address: string (nullable = true)
 |-- destroyed_amount_raw: long (nullable = true)
 |-- destroyed_amount_usdt: decimal(38,6) (nullable = true)
 |-- ingested_at: timestamp (nullable = true)


Total rows: 449,908

=== Sample ===
+------------+-------------------+----------+------------------------------------------------------------------+---------+------------------------------------------+-----+----------+------------------------------------------+---------------------------

## Lazy vs Action

Spark transformations are lazy - they do not execute immediately.

| Type | Examples | What happens |
|------|----------|--------------|
| Transformation | filter, select, groupBy, withColumn, repartition | Builds a plan (DAG); no data is read or computed yet |
| Action | show, count, collect, write, take, first | Triggers execution of the entire plan |

When you call an action, Spark:

1. Looks at the full chain of transformations
2. Optimizes it (Catalyst optimizer: push down filters, reorder joins, etc.)
3. Breaks it into stages and tasks
4. Sends tasks to executors

This is why you can chain 10 transformations cheaply - nothing happens until the first action.

**Spark UI checkpoint:** Run an action (count(), show()) and watch the Stages tab. Before the action, no stages exist. After the action, you see stages appear and tasks begin running.

In [ ]:
# These are transformations - LAZY, nothing executes yet
df_filtered = df.filter(F.col("token") == "USDC")
df_selected = df_filtered.select("event_date", "event_type", "amount_raw")
df_grouped = df_selected.groupBy("event_date").count()

print("Transformations built. Nothing has run yet.")
print("Spark UI Stages tab should still be empty.")

# This is an action - TRIGGERS execution
print("\n=== Calling count() (an ACTION) ===")
n = df_grouped.count()
print(f"Result: {n:,} rows")

print("\nNow check Spark UI -> Stages. You should see stages with tasks.")


## Union

docs/batch-processing/data/01_raw_data
docs/batch-processing/data/02_raw_data

In [9]:
# Read the second raw data folder
path2 = "data/02_raw_data"

df2 = (
    spark.read
    .option("recursiveFileLookup", "true")
    .parquet(path2)
)

# Union: stack df2 on top of df (same schema, match by column name)
df = df.unionByName(df2)

print("=== After Union ===")
print("Rows:", f"{df.count():,}")
df.printSchema()


=== After Union ===
Rows: 991,540
root
 |-- block_number: long (nullable = true)
 |-- block_timestamp: timestamp (nullable = true)
 |-- event_date: date (nullable = true)
 |-- transaction_hash: string (nullable = true)
 |-- log_index: integer (nullable = true)
 |-- contract_address: string (nullable = true)
 |-- token: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- topic1_address: string (nullable = true)
 |-- topic2_address: string (nullable = true)
 |-- amount_raw: string (nullable = true)
 |-- blacklisted_address: string (nullable = true)
 |-- destroyed_amount_raw: long (nullable = true)
 |-- destroyed_amount_usdt: decimal(38,6) (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



## Step 2 - Explore

Look at distinct values, filter by token or date, and count rows per category.

**Spark UI checkpoint:** distinct() and groupBy().count() trigger shuffle stages; filter() is narrow (no shuffle - each task filters its own files locally).

In [ ]:
print("=== Distinct event types ===")
df.select("event_type").distinct().show()

print("=== Distinct tokens ===")
df.select("token").distinct().show()

print("=== MINT events (first 5) ===")
df.filter(F.col("event_type") == "MINT").show(5, truncate=False)

print("=== BURN events (first 5) ===")
df.filter(F.col("event_type") == "BURN").show(5, truncate=False)

print("=== USDC events on 2026-05-07 ===")
df.filter(
    (F.col("token") == "USDC") &
    (F.col("event_date") == "2026-05-07")
).show(5, truncate=False)

print("=== Rows per event type ===")
df.groupBy("event_type").count().orderBy("count", ascending=False).show()

print("=== Rows per token ===")
df.groupBy("token").count().show()


## Step 3 - Transform

Derive new columns, cast types, and add business-logic flags.

**Key concept:** Transformations are lazy - printSchema() and show() are the actions that finally execute the plan.

In [14]:
df_t = (
    df
    # amount_raw is stored as string; cast to decimal for math
    .withColumn(
        "amount_usd",
        F.when(
            F.length(F.col("amount_raw")) <= 32,
            F.col("amount_raw").cast(DecimalType(38, 6)),
        ),
    )
    # Business flag: is this a mint?
    .withColumn("is_mint", F.col("event_type") == "MINT")
    # Business flag: is this a burn?
    .withColumn("is_burn", F.col("event_type") == "BURN")
    # Derive hour from timestamp for time-series analysis
    .withColumn("event_hour", F.date_trunc("hour", F.col("block_timestamp")))
)

print("=== Transformed sample ===")
df_t.select(
    "event_date", "event_hour", "token", "event_type",
    "amount_usd", "is_mint", "is_burn"
).show(5, truncate=False)

df_t.printSchema()


=== Transformed sample ===
+----------+-------------------+-----+----------+------------------+-------+-------+
|event_date|event_hour         |token|event_type|amount_usd        |is_mint|is_burn|
+----------+-------------------+-----+----------+------------------+-------+-------+
|2026-08-23|2026-08-23 11:00:00|USDC |BURN      |1174000000.000000 |false  |true   |
|2026-08-23|2026-08-23 11:00:00|USDC |BURN      |14715588436.000000|false  |true   |
|2026-08-23|2026-08-23 10:00:00|USDC |MINT      |4901274.000000    |true   |false  |
|2026-08-23|2026-08-23 10:00:00|USDC |MINT      |686.000000        |true   |false  |
|2026-08-23|2026-08-23 11:00:00|USDC |BURN      |652624.000000     |false  |true   |
+----------+-------------------+-----+----------+------------------+-------+-------+
only showing top 5 rows
root
 |-- block_number: long (nullable = true)
 |-- block_timestamp: timestamp (nullable = true)
 |-- event_date: date (nullable = true)
 |-- transaction_hash: string (nullable = true)

## Step 4 - Aggregate

Group by business dimensions and compute metrics.

**Spark UI checkpoint:** After groupBy, look at the Stages tab - you will see a shuffle stage where Spark exchanges data across partitions to aggregate by key.

In [16]:
print("=== Daily mint/burn volume by token ===")
daily = (
    df_t
    .groupBy("event_date", "token", "event_type")
    .agg(
        F.count("*").alias("event_count"),
        F.sum("amount_usd").alias("total_amount_usd"),
        F.avg("amount_usd").alias("avg_amount_usd")
    )
    .orderBy("event_date", "token", "event_type")
)

daily.show() #20, truncate=False)

# print("=== Net flow per token per day (mint - burn) ===")
# net = (
#     df_t
#     .groupBy("event_date", "token")
#     .agg(
#         F.sum(F.when(F.col("is_mint"), F.col("amount_usd")).otherwise(0)).alias("total_minted"),
#         F.sum(F.when(F.col("is_burn"), F.col("amount_usd")).otherwise(0)).alias("total_burned"),
#     )
#     .withColumn("net_flow", F.col("total_minted") - F.col("total_burned"))
#     .orderBy("event_date", "token")
# )

# net.show(20, truncate=False)


=== Daily mint/burn volume by token ===


+----------+-----+-----------+-----------+--------------------+--------------------+
|event_date|token| event_type|event_count|    total_amount_usd|      avg_amount_usd|
+----------+-----+-----------+-----------+--------------------+--------------------+
|2026-05-01| USDC|       BURN|         30|4589452742764.000000|152981758092.1333...|
|2026-05-01| USDC|       MINT|         53|3631241954947.000000|68513999149.94339...|
|2026-05-02| USDC|       BURN|       1242|101874167111521.0...|82024289139.71095...|
|2026-05-02| USDC|       MINT|       1646|91524361516779.00...|55604107847.37484...|
|2026-05-03| USDC|       BURN|       1297|95347278557903.00...|73513707446.34001...|
|2026-05-03| USDC|       MINT|       1643|61230826269664.00...|37267697060.05112...|
|2026-05-04| USDC|       BURN|       1638|563910872794876.0...|344267932109.2039...|
|2026-05-04| USDC|       MINT|       2301|809007120299413.0...|351589361277.4502...|
|2026-05-04| USDT|  BLACKLIST|          3|                NULL|  

## Repartition

Repartition changes the number of partitions in a DataFrame.

| | repartition(n) | coalesce(n) |
|---|------------------|---------------|
| Shuffle | Yes (full shuffle) | No (narrow, only reduces) |
| Can increase partitions | Yes | No |
| Can decrease partitions | Yes | Yes |
| Use when | You need more parallelism or even data distribution | You only need fewer files |

Repartition triggers a shuffle: data is read, redistributed across new partitions, and written again. You will see shuffle read/write in the Spark UI.

Common use: increase partitions before a heavy groupBy or join so each executor has more balanced work.

**Spark UI checkpoint:** Run repartition() and watch the Stages tab - you will see a shuffle write stage followed by a shuffle read stage.

In [19]:
# Check current number of partitions
print("Current partitions:", df.rdd.getNumPartitions())

# # Repartition to 20 partitions (triggers shuffle if increasing)
df_rep = df.repartition(20)
print("After repartition(20):", df_rep.rdd.getNumPartitions())

# # Run an action to force the shuffle
# print("\n=== Running count() after repartition (an ACTION) ===")
# n_rep = df_rep.count()
# print(f"Rows after repartition: {n_rep:,}")

print("\nNow check Spark UI -> Stages. You should see shuffle write + shuffle read stages.")


Current partitions: 4


After repartition(20): 20

Now check Spark UI -> Stages. You should see shuffle write + shuffle read stages.


## Step 5 - Spark SQL

Register the DataFrame as a temporary view and run SQL queries. Demo a broadcast join with a small lookup table.

**Key concept:** broadcast() sends the small lookup table to all executors, avoiding a shuffle for the join - critical when joining a big fact table with a small dimension table.

In [21]:
df_t.createOrReplaceTempView("issuer_events")

# Pure SQL aggregation
# spark.sql("""
#     SELECT
#         event_date,
#         token,
#         event_type,
#         COUNT(*) AS event_count,
#         SUM(amount_usd) AS total_amount_usd
#     FROM issuer_events
#     GROUP BY event_date, token, event_type
#     ORDER BY event_date DESC, total_amount_usd DESC
#     LIMIT 20
# """).show(truncate=False)

# Broadcast join with a small lookup table
tokens_lookup = spark.createDataFrame(
    [
        ("USDC", 6, "0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48"),
        ("USDT", 6, "0xdac17f958d2ee523a2206206994597c13d831ec7"),
    ],
    ["token", "decimals", "issuer_address"]
)

df_enriched = (
    df_t
    .join(F.broadcast(tokens_lookup), on="token", how="left")
)

df_enriched.select("token", "decimals", "issuer_address", "event_type", "amount_usd").show(5, truncate=False)


+-----+--------+------------------------------------------+----------+------------------+
|token|decimals|issuer_address                            |event_type|amount_usd        |
+-----+--------+------------------------------------------+----------+------------------+
|USDC |6       |0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48|BURN      |1174000000.000000 |
|USDC |6       |0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48|BURN      |14715588436.000000|
|USDC |6       |0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48|MINT      |4901274.000000    |
|USDC |6       |0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48|MINT      |686.000000        |
|USDC |6       |0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48|BURN      |652624.000000     |
+-----+--------+------------------------------------------+----------+------------------+
only showing top 5 rows


## Step 6 - Write Output

Write the transformed data back to Parquet. Two common patterns:

1. Partitioned output (recommended for analytics) - creates subdirectories like event_date=2026-05-07/token=USDC/
2. Single-file output (good for small reports) via coalesce(1)

**Spark UI checkpoint:** After write, check the Stages tab - you will see a write stage.

In [ ]:
# Non Partitioned write

import os

report_dir = "/home/freeman/Curriculum/docs/batch-processing/data/report"
os.makedirs(report_dir, exist_ok=True)

# Option A - Partitioned output (recommended for analytics)
daily = (
    df_t
    .groupBy("event_date", "token", "event_type")
    .agg(
        F.count("*").alias("event_count"),
        F.sum("amount_usd").alias("total_amount_usd"),
    )
)

daily.write.mode("overwrite").parquet(
    f"{report_dir}/daily_by_token/"
)


In [ ]:
# Partitioned write

import os

report_dir = "/home/freeman/Curriculum/docs/batch-processing/data/report"
os.makedirs(report_dir, exist_ok=True)

# Option A - Partitioned output (recommended for analytics)
daily = (
    df_t
    .groupBy("event_date", "token", "event_type")
    .agg(
        F.count("*").alias("event_count"),
        F.sum("amount_usd").alias("total_amount_usd"),
    )
)

daily.write.mode("overwrite").partitionBy("event_date",'token').parquet(
    f"{report_dir}/daily_by_token/"
)
